## Задание 1
Добавьте поддержку автоматической смешанной точности в уже знакомый вам код обучения. Используйте датасет и модель из фрагмента выше.

Дополните код ниже `AMP` с помощью `GradScaler` и `autocast`. Сравните время выполнения и потребление памяти с базовой версией.


In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

ds = load_dataset('zloelias/lenta-ru', split="train[:10%]")
model_name = 'sergeyzh/rubert-tiny-turbo'

tokenizer = AutoTokenizer.from_pretrained(model_name)
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = ds.map(preprocess_function, batched=True, 
                            remove_columns=['title', 'text', 'topic'])
data_collator = DataCollatorWithPadding(tokenizer, max_length=512, padding=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                                           num_labels=5, 
                                                           device_map='auto')

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 183.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: sergeyzh/rubert-tiny-turbo
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
# Добавьте необходимые импорты для AMP
# Ваш код здесь
from torch.amp import GradScaler, autocast 

num_epochs = 5
batch_size = 16
learning_rate = 5e-5

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# Инициализируйте GradScaler
# Ваш код здесь
scaler = GradScaler()

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        # Переносим тензоры на device
        batch = {k: v.to(model.device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        # Добавьте autocast контекст и модифицируйте forward pass
        # Ваш код здесь
        with autocast(device_type='cuda', dtype=torch.float16): 
            outputs = model(**batch)
            loss = outputs.loss

        # Модифицируйте backward pass и optimizer step для AMP
        # Ваш код здесь
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()    
        
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")

/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_40710/1800976079.py:22: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = amp.GradScaler()
Epoch 1:   0%|          | 0/1163 [00:00<?, ?it/s]/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv312/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_40710/1800976079.py:35: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
Epoch 1:   8%|▊         | 93/1163 [02:05<24:04,  1.35s/it] 


RuntimeError: MPS backend out of memory (MPS allocated: 10.58 GiB, other allocations: 9.22 GiB, max allowed: 20.13 GiB). Tried to allocate 373.39 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## Задание 2
Добавьте аккумуляцию градиентов в базовый код обучения. Это позволит имитировать большие размеры батчей при ограниченной памяти GPU.

Дополните код ниже реализацией аккумуляции градиентов с `accumulation_steps=4`. Эффективным размером батча станет 64 (16 × 4). Обновите параметры с каждым `accumulation_steps`, если остаются необработанные градиенты, проверяя с помощью `if n_batches % accumulation_steps != 0:`.

In [3]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

num_epochs = 5
batch_size = 16  # Реальный batch size
learning_rate = 5e-5
accumulation_steps = 4  # Эффективный batch size = 16 * 4 = 64

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    n_batches = 0

    optimizer.zero_grad()
        
    for i, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}")):
        # Переносим тензоры на device
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        # Ваш код: масштабируйте loss для аккумуляции
        loss = loss / accumulation_steps
                
        loss.backward()
        
        # Ваш код: логика обновления параметров каждые accumulation_steps
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps  # Не забудьте учесть масштабирование при подсчёте
        n_batches += 1

    # Обновляем параметры, если остались необработанные градиенты
    if n_batches % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()
    
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    print(f"Epoch {epoch+1} effective optimizer steps: {n_batches // accumulation_steps}")

Epoch 1:   0%|          | 0/1163 [00:00<?, ?it/s]/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv312/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Epoch 1:   0%|          | 3/1163 [00:03<24:50,  1.28s/it]


RuntimeError: MPS backend out of memory (MPS allocated: 10.57 GiB, other allocations: 9.35 GiB, max allowed: 20.13 GiB). Tried to allocate 291.62 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

Объедините автоматическую смешанную точность и аккумуляцию градиентов в коде ниже.

In [ ]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
# Ваши импорты для AMP
from torch.amp import GradScaler, autocast 


num_epochs = 5
batch_size = 16
learning_rate = 5e-5
accumulation_steps = 4

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Ваш код: инициализация для AMP
scaler = GradScaler()

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    n_batches = 0
    
    # Ваш код: управление градиентами
    for i, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}")):
        # Переносим тензоры на device
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with autocast(device_type='cuda', dtype=torch.float16): 
            outputs = model(**batch)
            loss = outputs.loss / accumulation_steps
                
        scaler.scale(loss).backward()
        
        # Ваш код: логика обновления параметров каждые accumulation_steps
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps  # Не забудьте учесть масштабирование при подсчёте
        n_batches += 1

    # Обновляем параметры, если остались необработанные градиенты
    if n_batches % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")

/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_40710/3037288650.py:23: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/1163 [00:00<?, ?it/s]/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv312/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/var/folders/_4/p98h_g953379cg5tzm5_l5p40000gn/T/ipykernel_40710/3037288650.py:34: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast(device_type='cuda', dtype=torch.float16):
Epoch 1:   0%|          | 2/1163 [00:03<29:07,  1.51s/it]


RuntimeError: MPS backend out of memory (MPS allocated: 10.57 GiB, other allocations: 9.44 GiB, max allowed: 20.13 GiB). Tried to allocate 189.01 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).